In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch import nn
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_curve
)

import matplotlib.pyplot as plt

print("Libraries loaded successfully.")

/Users/Mourya/miniforge3/envs/aircraft-conflict-gnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully.


In [2]:
train_graphs = torch.load(
    "processed_data/train_graphs.pt",
    weights_only=False
)

val_graphs = torch.load(
    "processed_data/val_graphs.pt",
    weights_only=False
)

test_graphs = torch.load(
    "processed_data/test_graphs.pt",
    weights_only=False
)

print("Train:", len(train_graphs))
print("Validation:", len(val_graphs))
print("Test:", len(test_graphs))

Train: 504
Validation: 108
Test: 108


In [4]:
train_loader = DataLoader(
    train_graphs,
    batch_size=8,
    shuffle=False
)

val_loader = DataLoader(
    val_graphs,
    batch_size=8,
    shuffle=False
)

test_loader = DataLoader(
    test_graphs,
    batch_size=8,
    shuffle=False
)

print("DataLoaders ready.")

DataLoaders ready.


In [5]:
from pathlib import Path

print("GAT model exists:",
      Path("models/conflict_gat_large_best.pth").exists())

print("GCN model exists:",
      Path("models/conflict_gcn.pth").exists())

GAT model exists: True
GCN model exists: True


In [6]:
from torch import nn
from torch_geometric.nn import GATConv
import torch.nn.functional as F

class ConflictGAT(nn.Module):

    def __init__(self,
                 node_dim=6,
                 edge_dim=5,
                 hidden_dim=64,
                 heads=4):

        super().__init__()

        self.gat1 = GATConv(
            node_dim,
            hidden_dim,
            heads=heads,
            concat=True
        )

        self.gat2 = GATConv(
            hidden_dim * heads,
            hidden_dim,
            heads=1,
            concat=False
        )

        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + edge_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, data):

        x = F.elu(self.gat1(data.x, data.edge_index))
        x = self.gat2(x, data.edge_index)

        src = x[data.edge_index[0]]
        dst = x[data.edge_index[1]]

        edge_input = torch.cat(
            [src, dst, data.edge_attr],
            dim=1
        )

        return self.edge_mlp(edge_input).squeeze(-1)

In [8]:
from torch import nn
from torch_geometric.nn import GATConv
import torch.nn.functional as F

class ConflictGAT(nn.Module):

    def __init__(self):

        super().__init__()

        self.gat1 = GATConv(
            6,
            64,
            heads=2,
            concat=True
        )

        self.gat2 = GATConv(
            128,
            64,
            heads=1,
            concat=False
        )

        self.edge_mlp = nn.Sequential(
            nn.Linear(64*2 + 5, 64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

    def forward(self,data):

        x = F.elu(self.gat1(data.x,data.edge_index))
        x = self.gat2(x,data.edge_index)

        src = x[data.edge_index[0]]
        dst = x[data.edge_index[1]]

        edge_input = torch.cat(
            [src,dst,data.edge_attr],
            dim=1
        )

        return self.edge_mlp(edge_input).squeeze(-1)

In [10]:
state = torch.load(
    "models/conflict_gat_large_best.pth",
    map_location="cpu",
    weights_only=False
)

for k, v in state.items():
    print(k, tuple(v.shape))

gat1.att_src (1, 2, 64)
gat1.att_dst (1, 2, 64)
gat1.att_edge (1, 2, 64)
gat1.bias (128,)
gat1.lin.weight (128, 6)
gat1.lin_edge.weight (128, 5)
gat2.att_src (1, 1, 64)
gat2.att_dst (1, 1, 64)
gat2.att_edge (1, 1, 64)
gat2.bias (64,)
gat2.lin.weight (64, 128)
gat2.lin_edge.weight (64, 5)
edge_mlp.0.weight (64, 133)
edge_mlp.0.bias (64,)
edge_mlp.2.weight (1, 64)
edge_mlp.2.bias (1,)


In [11]:
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class ConflictGAT(nn.Module):

    def __init__(self):
        super().__init__()

        self.gat1 = GATConv(
            in_channels=6,
            out_channels=64,
            heads=2,
            concat=True,
            edge_dim=5
        )

        self.gat2 = GATConv(
            in_channels=128,
            out_channels=64,
            heads=1,
            concat=False,
            edge_dim=5
        )

        self.edge_mlp = nn.Sequential(
            nn.Linear(64*2 + 5, 64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

    def forward(self,data):

        x = F.elu(
            self.gat1(
                data.x,
                data.edge_index,
                data.edge_attr
            )
        )

        x = self.gat2(
            x,
            data.edge_index,
            data.edge_attr
        )

        src = x[data.edge_index[0]]
        dst = x[data.edge_index[1]]

        edge_input = torch.cat(
            [src,dst,data.edge_attr],
            dim=1
        )

        return self.edge_mlp(edge_input).squeeze(-1)

In [ ]:
device = torch.device("cpu")

gat_model = ConflictGAT().to(device)

state = torch.load(
    "models/conflict_gat_large_best.pth",
    map_location=device,
    weights_only=False
)

gat_model.load_state_dict(state)

gat_model.eval()

print("✅ GAT loaded successfully")